<a href="https://colab.research.google.com/github/bjanesko/ComputationalChemistryForExperimentalists/blob/main/Chapter10Part2_MachineLearningClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Computational Chemistry for Experimentalists**

**Chapter 10 part 2: Machine learning**

Artificial intelligence,  machine learning, and chemical databases are rapidly transforming the chemical enterprise. This example illustrates building a machine-learned classifier for blood-brain barrier permeability. The classifier and dataset and framework are taken directly from the MoleculeNet repository of chemical machine learning bechmarks. See in particular

https://github.com/deepchem/moleculenet/blob/master/examples/fingerprint.py

This first block imports all of the necessary Python modules. If these aren't installed, this will fail. Ignore the warnings from the "under testing" PySCF properties modules.

In [ ]:
!pip install deepchem
!pip install hyperopt
!pip install rdkit
!pip install py3Dmol
import os
import py3Dmol
import warnings
import deepchem as dc
import pandas as pd
import json
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
from deepchem.molnet import load_bbbp

from copy import deepcopy
from functools import partial
from hyperopt import hp, fmin, tpe
from shutil import copyfile
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

The next block is AI-generated code that reads in the  BBBP dataset of blood-brain barrier permeabilities. The blood-brain barrier blocks most molecules from crossing from the blood into the brain. The BBBP dataset is a list of over 2000 molecules represented by their SMILES string and classified for significant permeation across the BBB. The dataset contains two columns: the molecule SMILES string, and the 'p_np' classification of BBB permeability.

Technical note: The original MoleculeNet example throws errors due to nonparsing SMILES strings in the version of RDKit I'm using. This version does strict filtering of the SMILES strings before passing to MoleculeNet.

In [ ]:

warnings.filterwarnings("ignore", message=".*please use MorganGenerator.*")
RDLogger.DisableLog('rdApp.*')

# 2. Setup paths and download the raw BBBP dataset directly
data_dir = dc.utils.data_utils.get_data_dir()
dataset_file = os.path.join(data_dir, "BBBP.csv")

if not os.path.exists(dataset_file):
    print("Downloading raw BBBP dataset...")
    dc.utils.data_utils.download_url(
        url="https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv",
        dest_dir=data_dir
    )

# 3. Load into Pandas and strictly filter out everything that fails SMILES parsing
df = pd.read_csv(dataset_file)
print(f"Original dataset rows: {len(df)}")

def is_valid_smiles(smiles):
    if not isinstance(smiles, str):
        return False
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None

# Apply strict structural filtering
df_clean = df[df['smiles'].apply(is_valid_smiles)].copy()
print(f"Cleaned dataset rows:  {len(df_clean)}")
print(f"Cleaned dataset fields: ")
print(df_clean.columns.tolist)

Original dataset rows: 2050
Cleaned dataset rows:  2039
Cleaned dataset fields: 
<bound method IndexOpsMixin.tolist of Index(['num', 'name', 'p_np', 'smiles'], dtype='object')>


The next block illustrates a few structures from the BBBP dataset, converting the SMILES string representation to a chemical structure representation using the RDKit. All four are classified as nonpermeating

In [ ]:
print(df_clean['p_np'][102:106])

104    0
105    0
106    0
107    0
Name: p_np, dtype: int64


In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df_clean['smiles'][102:106]]
p = py3Dmol.view(width=600,height=200,viewergrid=(2,2))
for i in range(2):
  for j in range(2):
    m2=Chem.AddHs(mols[i*2+j])
    AllChem.EmbedMolecule(m2)
    AllChem.MMFFOptimizeMolecule(m2)
    p.addModel(Chem.MolToMolBlock(m2), 'sdf',viewer=(j,i))
    p.setStyle({'stick':{},'sphere':{"scale":0.3}},viewer=(j,i))
p.render()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

The nexxt step is to generate a Morgan fingerprint for each structure. A Morgan fingerprint is a compact 2048-bit binary vector represeting a chemical structure as a list of atoms, a list of nearest neighbor atom pairs, a list of next-nearest-neighbors, and so on. I'll first illustrate the Morgan fingerprint of benzene molecule. This turns out to be an array of mostly zeros, with only a few ones.

In [ ]:
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
mol = Chem.MolFromSmiles('c1ccccc1')
fp = generator.GetFingerprintAsNumPy(mol).astype(int)
for i in range(0, len(fp), 100):
    chunk = fp[i : i + 100]
    print("".join(str(int(x)) for x in chunk))

0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000010000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

The code block below generates the Morgan fingerprints, and then uses Molecule

In [ ]:
# 4. Generate RDKit Morgan Fingerprints for each
print("\nFeaturizing clean structures via modern Morgan Generator...")
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

features = []
for smiles in df_clean['smiles']:
    mol = Chem.MolFromSmiles(smiles)
    fp = generator.GetFingerprintAsNumPy(mol).astype(np.float32)
    features.append(fp)

X = np.array(features)
y = df_clean['p_np'].to_numpy().reshape(-1, 1)  # BBBP target tasks matrix
w = np.ones_like(y)                             # Missing/weights mask (1 for all clean valid rows)
ids = df_clean['smiles'].to_numpy()

# 5. Pack directly into a compatible DeepChem DiskDataset object
# This removes any dependency on internal data loader structures
dataset = dc.data.DiskDataset.from_numpy(X=X, y=y, w=w, ids=ids)



Featurizing clean structures via modern Morgan Generator...


Machine learning models, like linear fits, need to be fit to data. A key part of validation is to split the training data into "fitting" and "testing" subsets. One fits the model parameters on the fitting data, then tests to see if it will extrapolate successfully to testing data. This is usually a three-sstep process: fit the parameters to training data, adjust the 'hyperparameters' (e.g. neural net architecture) to a second set of validation data, then test the final model on a third unseen set of testing data. The code below uses MoleculeNet routines to split up the data into training, validation, and testing subsets. 80% of the data is in the training set, the remainder is split into validation and testing sets.

In [ ]:
# 6. Apply standard MoleculeNet Scaffold Split safely
print("Applying Scaffold Split...")
splitter = dc.splits.ScaffoldSplitter()
train_dataset, valid_dataset, test_dataset = splitter.train_valid_test_split(dataset)

print("\n--- Pipeline Completed Successfully ---")
print(f"Train Dataset Shape:  {train_dataset.X.shape}")
print(f"Valid Dataset Shape:  {valid_dataset.X.shape}")
print(f"Test Dataset Shape:   {test_dataset.X.shape}")

Applying Scaffold Split...

--- Pipeline Completed Successfully ---
Train Dataset Shape:  (1631, 2048)
Valid Dataset Shape:  (204, 2048)
Test Dataset Shape:   (204, 2048)


The next step is to fit the model on the training dataset. I'll use a random forest classifier as implemented in scikit-learn , with evaluation metric being ROC-AIC

In [ ]:
# 1. Initialize the Scikit-Learn classifier
sklearn_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Wrap it in DeepChem's SklearnModel
model = dc.models.SklearnModel(sklearn_rf)

# 3. Fit the model to your dataset
model.fit(train_dataset)

# 4. Define evaluation metrics (e.g., ROC-AUC and Accuracy)
metrics = [
    dc.metrics.Metric(dc.metrics.roc_auc_score),
    dc.metrics.Metric(dc.metrics.accuracy_score)
]

# 5. Evaluate the model
train_scores = model.evaluate(train_dataset, metrics)
test_scores = model.evaluate(test_dataset, metrics)

print("Train scores:", train_scores)
print("Test scores:", test_scores)

Train scores: {'roc_auc_score': np.float64(0.9997737149322431), 'accuracy_score': np.float64(0.9920294297976702)}
Test scores: {'roc_auc_score': np.float64(0.6905530397918874), 'accuracy_score': np.float64(0.5882352941176471)}


Finally, we'll apply our model to methane, cyclohexane, benzene, methanol, asprin, and the four BBBP-nonpermeating molecules from the original dataset  shown above. For each we get a probability of it being classified as BBB penetrating. The small molecules are predicted as highly permeating, aspirin is predicted about 50%  permeating, and the test cases from the original datest are mostly treated OK.

In [ ]:
smi_list = ['C','C1CCCCC1','c1ccccc1','CO','CC(=O)OC1=CC=CC=C1C(=O)O']
smi_list.extend(df_clean['smiles'][102:106])
for smi in(smi_list):
  mol = Chem.MolFromSmiles(smi)
  fp = generator.GetFingerprintAsNumPy(mol).astype(np.float32)
  single_datapoint_dataset = dc.data.NumpyDataset(X=[fp])
  probabilities = model.predict(single_datapoint_dataset)
  print('BBBP probability ',probabilities[0][1],smi)

BBBP probability  0.95 C
BBBP probability  1.0 C1CCCCC1
BBBP probability  0.98 c1ccccc1
BBBP probability  0.99 CO
BBBP probability  0.5737857142857142 CC(=O)OC1=CC=CC=C1C(=O)O
BBBP probability  0.7366666666666666 CCCCc1oc2ccccc2c1C(=O)c3cc(I)c(OCCN(CC)CC)c(I)c3
BBBP probability  0.0 O.O.O.CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)c3ccc(O)cc3)C(=O)N2[C@H]1C(O)=O
BBBP probability  0.0 CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)c3ccccc3)C(=O)N2[C@H]1C(O)=O
BBBP probability  0.29 C[C@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@]4(C)[C@H]3CC[C@]12C


Practice Problems:

1) Test if changing the random forest classifier's hyperparameters (n_estimators, random_state, etc) can improve the performance for the training and testing sets.

2) Choose one molecule relevant to your ongoing research or your research interests, preferably one that has some biochemical relevance. Evaluate your model's prediction for that molecule BBB permeability. Does the result make physical sense? Explain

3) (HARD/extra credit). Change your fitting procedure to use RDKit moleular graphs saved as PyTorch Geometric data objects as your descriptors, then fit the data to those descriptors using some kind of graph neural network. You may consider referring to the qcMol project, which further extends this to use quantum-mechanical graph features

https://github.com/GHUSER-haoyu/qcMol/tree/main$0